# 💳 UPI Transaction Monitoring & Banking Dashboards
**Author:** Gurupriya R | Mphasis

5-panel dashboard for monitoring UPI payments, EMI tracking, anomaly detection, and merchant settlements.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
df = pd.read_csv('data/transactions.csv')
df['timestamp'] = pd.to_datetime(df['timestamp'])
df['hour'] = df['timestamp'].dt.hour
print(f'{len(df):,} transactions | Success rate: {(df["status"]=="Success").mean():.1%}')

## 1. Transaction Overview

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
df['transaction_type'].value_counts().plot(kind='pie', ax=axes[0], autopct='%1.0f%%',
    colors=['#e8924a','#3a8c8c','#5ab0b0','#e07060','#c4712e'])
axes[0].set_title('Transaction Type Mix')
df['status'].value_counts().plot(kind='bar', ax=axes[1],
    color=['#5ab0b0','#e07060','#e8924a'], edgecolor='white')
axes[1].set_title('Transaction Status')
df.groupby('bank')['amount'].sum().sort_values().plot(kind='barh', ax=axes[2], color='#3a8c8c')
axes[2].set_title('Volume by Bank (₹)')
plt.tight_layout(); plt.show()

## 2. Anomaly Detection

In [ ]:
mean_amt, std_amt = df['amount'].mean(), df['amount'].std()
df['is_anomaly'] = (df['amount'] > mean_amt + 3*std_amt).astype(int)
print(f'Anomalies detected: {df["is_anomaly"].sum()} ({df["is_anomaly"].mean():.1%})')
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(df[df['is_anomaly']==0]['amount'], bins=50, alpha=0.7, color='#5ab0b0', label='Normal')
ax.hist(df[df['is_anomaly']==1]['amount'], bins=20, alpha=0.9, color='#e07060', label='Anomaly')
ax.set_title('Transaction Amount Distribution — Anomaly Detection')
ax.legend(); ax.set_xlabel('Amount (₹)')
plt.tight_layout(); plt.show()

## 3. Hourly Pattern & EMI Tracking

In [ ]:
hourly = df.groupby('hour')['amount'].agg(['count','sum'])
emi_df = df[df['transaction_type']=='EMI Debit']
fig, axes = plt.subplots(1,2,figsize=(14,4))
hourly['count'].plot(ax=axes[0], color='#e8924a', marker='o', linewidth=2)
axes[0].set_title('Hourly Transaction Count')
axes[0].set_xlabel('Hour of Day')
emi_df.groupby('bank')['emi_overdue'].mean().sort_values().plot(kind='barh', ax=axes[1], color='#e07060')
axes[1].set_title('EMI Overdue Rate by Bank')
plt.tight_layout(); plt.show()